In [1]:
import os
import requests
import xml.etree.ElementTree as ET
import gradio as gr
from dotenv import load_dotenv
from agents import (
    Agent, Runner, function_tool,
    set_default_openai_api, set_tracing_disabled,
)

c:\Users\DELL\Documents\Programming\Python\Agentic\Chatbot for Sports_Pol_wea\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)
token = os.getenv("API_TOKEN")
if not token:
    raise ValueError("API_TOKEN not found. Check your .env file.")

In [3]:
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

In [4]:
MODEL = "openai/gpt-4o-mini"
TIMEOUT = 10
HEADERS = {"User-Agent": "NewsChatbot/1.0"}

In [5]:
WEATHER_CODES = {
    0: "Clear sky", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle",
    61: "Light rain", 63: "Rain", 65: "Heavy rain", 71: "Light snow",
    73: "Snow", 75: "Heavy snow", 80: "Rain showers", 81: "Rain showers",
    82: "Violent rain showers", 95: "Thunderstorm", 96: "Thunderstorm with hail",
    99: "Thunderstorm with hail",
}

In [6]:
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for any city in the world.

    Args:
        city: The city name, e.g. "Abuja" or "London"
    """
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            headers=HEADERS, timeout=TIMEOUT,
        ).json()
        if not geo.get("results"):
            return f"I couldn't find a city called {city}."

        place = geo["results"][0]
        data = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
            },
            headers=HEADERS, timeout=TIMEOUT,
        ).json()["current"]

        sky = WEATHER_CODES.get(data["weather_code"], "Unknown conditions")
        return (
            f"{place['name']}, {place.get('country', '')}: {sky}, "
            f"{data['temperature_2m']}°C, humidity {data['relative_humidity_2m']}%, "
            f"wind {data['wind_speed_10m']} km/h"
        )
    except Exception as e:
        return f"The weather service is unavailable right now ({e})."

In [7]:
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for any city in the world.

    Args:
        city: The city name, e.g. "Abuja" or "London"
    """
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            headers=HEADERS, timeout=TIMEOUT,
        ).json()
        if not geo.get("results"):
            return f"I couldn't find a city called {city}."

        place = geo["results"][0]
        data = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
            },
            headers=HEADERS, timeout=TIMEOUT,
        ).json()["current"]

        sky = WEATHER_CODES.get(data["weather_code"], "Unknown conditions")
        return (
            f"{place['name']}, {place.get('country', '')}: {sky}, "
            f"{data['temperature_2m']}°C, humidity {data['relative_humidity_2m']}%, "
            f"wind {data['wind_speed_10m']} km/h"
        )
    except Exception as e:
        return f"The weather service is unavailable right now ({e})."

In [8]:
NEWS_FEEDS = {
    "get_sports_news": {
        "description": "Get the latest headlines across all sports.",
        "url": "https://feeds.bbci.co.uk/sport/rss.xml",
    },
    "get_football_news": {
        "description": "Get the latest football (soccer) headlines.",
        "url": "https://feeds.bbci.co.uk/sport/football/rss.xml",
    },
    "get_politics_news": {
        "description": "Get the latest UK politics headlines.",
        "url": "https://feeds.bbci.co.uk/news/politics/rss.xml",
    },
    "get_world_news": {
        "description": "Get the latest world news and international politics headlines.",
        "url": "https://feeds.bbci.co.uk/news/world/rss.xml",
    },
    "get_africa_news": {
        "description": "Get the latest news and politics headlines from Africa.",
        "url": "https://feeds.bbci.co.uk/news/world/africa/rss.xml",
    },
}

In [9]:
def make_news_tool(name: str, description: str, url: str):
    @function_tool(name_override=name, description_override=description)
    def news_tool(limit: int) -> str:
        """Fetch headlines.

        Args:
            limit: How many headlines to return, from 1 to 10
        """
        try:
            limit = max(1, min(int(limit), 10))
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            items = ET.fromstring(response.content).iter("item")

            lines = []
            for item in items:
                title = item.findtext("title", "").strip()
                summary = item.findtext("description", "").strip()
                link = item.findtext("link", "").strip()
                lines.append(f"- {title}: {summary} ({link})")
                if len(lines) == limit:
                    break
            return "\n".join(lines) or "No headlines found right now."
        except Exception as e:
            return f"The news feed is unavailable right now ({e})."

    return news_tool

In [10]:
news_tools = [
    make_news_tool(name, cfg["description"], cfg["url"])
    for name, cfg in NEWS_FEEDS.items()
]

ALL_TOOLS = [get_weather] + news_tools

In [11]:
instructions = """You are a friendly assistant for sports news, politics news, and weather.

How to work:
- Weather question -> use get_weather.
- Sports question -> use get_sports_news, or get_football_news for football.
- Politics question -> use get_politics_news for the UK, get_africa_news for Africa,
  and get_world_news for anything international.
- If the user asks for several things at once (for example weather AND sports),
  call all the tools you need, then combine the results into one clear reply.
- Keep answers short and easy to scan. Include the link for each news story you mention.
- On politics, stay neutral. Report what happened and who said what. Never give your own
  opinion or tell the user who to support.
- If a tool fails or returns nothing, say so honestly. Never invent headlines or weather.
- If asked about something outside sports, politics, or weather, politely say what you can help with.
"""

In [12]:
assistant = Agent(
    name="NewsWeatherAssistant",
    instructions=instructions,
    model=MODEL,
    tools=ALL_TOOLS,
)

In [13]:
async def live_chat(message, history):
    conversation = [{"role": h["role"], "content": h["content"]}
                    for h in history]
    conversation.append({"role": "user", "content": message})
    result = await Runner.run(assistant, conversation)
    return result.final_output

In [14]:
gr.ChatInterface(
    live_chat,
    title="Sports, Politics & Weather Assistant",
    description="Ask me for the latest sports news, politics headlines, or the weather in any city.",
    examples=[
        "What's the weather in Abuja?",
        "Any big football news today?",
        "Give me Africa politics headlines and the weather in Lagos",
    ],
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
